# Comparing the models and choosing a threshold

A visual companion to the model comparison, the tuning and the threshold
analysis. Everything here uses the **training set only**; the test set is not
touched.

1. The default models, on every metric.
2. Default against tuned settings.
3. The four tuned models: precision-recall and ROC curves, metrics against the
   threshold, how well the predicted probabilities separate the classes, and
   confusion matrices at the candidate operating points.
4. An interactive threshold slider.

Run `uv run python -m machine_failure.data` first to download the data. The notebook
takes about two minutes to run (the default comparison and the out-of-fold
predictions).

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from sklearn.metrics import roc_curve

from machine_failure.baseline import METRICS, compare_models
from machine_failure.data import load_xy, split_train_test
from machine_failure.models import build_all_models
from machine_failure.predictions import out_of_fold_predictions
from machine_failure.thresholds import (
    DEFAULT_TARGETS,
    best_f1_per_model,
    compare_at_recall_targets,
    threshold_curve,
)
from machine_failure.tuned_models import build_tuned_models

DATA_PATH = Path("..") / "data" / "raw" / "ai4i2020.csv"
X, y = load_xy(DATA_PATH)
X_train, _, y_train, _ = split_train_test(X, y)  # the test set is not used here
print(f"training set: {len(X_train):,} rows, {int(y_train.sum())} failures")

In [ ]:
# The first four slots of the categorical palette, in order, plus neutral ink.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "text.color": INK,
        "axes.labelcolor": INK_MUTED,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "legend.frameon": False,
        "figure.dpi": 100,
    }
)

## 1. The default models

Each model with default settings, scored with 5-fold stratified cross-validation
on the training set. Bars show the mean across folds and the whiskers one
standard deviation; the models are ordered by PR-AUC. Precision, recall and F1
use the default threshold of 0.5.

In [ ]:
default_table = compare_models(build_all_models(), X_train, y_train)
default_table[("pr_auc", "mean")].sort_values(ascending=False).round(3).to_frame()

In [ ]:
order = default_table[("pr_auc", "mean")].sort_values().index  # best at the top
titles = {
    "pr_auc": "PR-AUC (the main metric)",
    "roc_auc": "ROC-AUC",
    "recall": "Recall",
    "precision": "Precision",
    "f1": "F1",
    "accuracy": "Accuracy",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, metric in zip(axes.flat, METRICS, strict=True):
    means = default_table.loc[order, (metric, "mean")]
    stds = default_table.loc[order, (metric, "std")]
    ax.barh(order, means, xerr=stds, color=BLUE, height=0.65, ecolor=INK_MUTED)
    ax.set_xlim(0, 1.05)
    ax.set_title(titles[metric])
    ax.grid(axis="y", visible=False)
fig.suptitle(
    "Default models, mean and standard deviation over 5 folds", x=0.01, ha="left"
)
fig.tight_layout()
plt.show()

Accuracy is high for every model, even the dummy that always predicts "no
failure", which is why it is not used to compare them. ROC-AUC barely separates
the best four, while PR-AUC does.

## 2. Default against tuned settings

The four best models were tuned with a randomized search (20 settings each) and
scored with **nested cross-validation**, which is an honest estimate because the
search never sees the fold it is scored on. That run takes about 20 minutes
(`uv run python -m machine_failure.tune`), so the numbers below are copied from
the README and not recomputed here.

In [ ]:
# PR-AUC (mean, std) over the same 5 outer folds, from `python -m machine_failure.tune`
NESTED = {
    "random_forest": ((0.890, 0.030), (0.911, 0.025)),
    "hist_gradient_boosting": ((0.889, 0.017), (0.893, 0.022)),
    "extra_trees": ((0.824, 0.026), (0.873, 0.028)),
    "xgboost": ((0.855, 0.029), (0.870, 0.034)),
}

names = list(NESTED)[::-1]  # best at the top
positions = np.arange(len(names))
fig, ax = plt.subplots(figsize=(8, 4))
for offset, index, color, label in [
    (0.15, 0, BLUE, "default"),
    (-0.15, 1, ORANGE, "tuned"),
]:
    means = np.array([NESTED[name][index][0] for name in names])
    stds = np.array([NESTED[name][index][1] for name in names])
    ax.errorbar(
        means,
        positions + offset,
        xerr=stds,
        fmt="o",
        color=color,
        markersize=7,
        capsize=3,
        label=label,
    )
    for position, mean, std in zip(positions + offset, means, stds, strict=True):
        ax.text(
            mean + std + 0.006,
            position,
            f"{mean:.3f}",
            va="center",
            color=INK,
            fontsize=9,
        )
ax.set_yticks(positions, names)
ax.set_xlim(0.78, 0.99)
ax.set_xlabel("PR-AUC (dot: mean, whisker: one standard deviation)")
ax.set_title("Tuning helps modestly; extra trees gains the most")
ax.legend(loc="lower left")
ax.grid(axis="y", visible=False)
plt.show()

## 3. The tuned models, out of fold

For each of the four tuned models, every training row gets one failure
probability from a model that never saw that row (5-fold). Everything below is
computed from those 8,000 probabilities per model.

In [ ]:
oof = {
    name: out_of_fold_predictions(model, X_train, y_train)
    for name, model in build_tuned_models().items()
}
curves = {
    name: threshold_curve(frame["label"], frame["probability"])
    for name, frame in oof.items()
}
colors = dict(zip(oof, [BLUE, ORANGE, AQUA, YELLOW], strict=True))
pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in oof.items()},
        "failures": {name: int(frame["label"].sum()) for name, frame in oof.items()},
    }
)

### Precision-recall and ROC curves

The dotted lines mark the recall targets (80%, 85% and 90%). On the left, read
the precision where a curve crosses a line: that is the share of alerts that are
real failures at that recall. The ROC curves on the right are nearly
indistinguishable, because the false positive rate divides by the thousands of
normal rows, which hides how many false alarms there are.

In [ ]:
fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(13, 5))
for name, frame in oof.items():
    curve = curves[name]
    ax_pr.plot(
        curve["recall"], curve["precision"], color=colors[name], lw=2, label=name
    )
    fpr, tpr, _ = roc_curve(frame["label"], frame["probability"])
    ax_roc.plot(fpr, tpr, color=colors[name], lw=2, label=name)
for target in DEFAULT_TARGETS:
    ax_pr.axvline(target, color=INK_MUTED, lw=1, ls=":")
    ax_pr.text(target, 0.02, f"{target:.0%}", rotation=90, ha="right", color=INK_MUTED)
ax_pr.set_xlabel("Recall (share of failures caught)")
ax_pr.set_ylabel("Precision (share of alerts that are real)")
ax_pr.set_title("Precision against recall")
ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(0, 1.02)
ax_pr.legend(loc="lower left")
ax_roc.plot([0, 1], [0, 1], color=INK_MUTED, lw=1, ls="--")
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate (recall)")
ax_roc.set_title("ROC")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.02)
fig.tight_layout()
plt.show()

### Precision, recall and F1 against the threshold

The threshold axis is logarithmic, because the interesting thresholds for high
recall are small (some are below 0.1). The dotted line marks the threshold with
the best F1. Moving left (a lower threshold) flags more machines: recall rises
and precision falls.

In [ ]:
best = best_f1_per_model(oof)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, name in zip(axes.flat, oof, strict=True):
    curve = curves[name]
    ax.plot(curve["threshold"], curve["precision"], color=BLUE, lw=2, label="precision")
    ax.plot(curve["threshold"], curve["recall"], color=ORANGE, lw=2, label="recall")
    ax.plot(curve["threshold"], curve["f1"], color=INK, lw=1.5, ls="--", label="F1")
    ax.axvline(best.loc[name, "threshold"], color=INK_MUTED, lw=1, ls=":")
    ax.set_xscale("log")
    ax.set_xlim(0.005, 1)
    ax.set_ylim(0, 1.02)
    ax.set_title(f"{name}  (best F1 at {best.loc[name, 'threshold']:.3f})")
axes[0, 0].legend(loc="lower left")
for ax in axes[1]:
    ax.set_xlabel("Threshold (log scale)")
fig.tight_layout()
plt.show()

The best thresholds differ a lot between the models (about 0.48 for the random
forest, 0.86 for gradient boosting). That is expected: gradient boosting and the
other class-weighted models give failures extra weight when they train, which
pushes all their probabilities up. So **compare models at the same recall (or the
same precision), not at the same threshold**; each model has its own scale.

### How well the probabilities separate the classes

Each panel shows the predicted failure probability of the normal rows (blue)
and of the real failures (orange), each histogram scaled to the same area. The
vertical axis is logarithmic. A good model puts the failures on the right and the
normal rows on the left. The orange mass near zero is the failures the model
cannot see, which is where the cost of a high recall target comes from.

In [ ]:
bins = np.linspace(0, 1, 41)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, (name, frame) in zip(axes.flat, oof.items(), strict=True):
    normal = frame.loc[frame["label"] == 0, "probability"]
    failed = frame.loc[frame["label"] == 1, "probability"]
    ax.hist(normal, bins=bins, density=True, color=BLUE, alpha=0.55)
    ax.hist(failed, bins=bins, density=True, color=ORANGE, alpha=0.55)
    ax.set_yscale("log")
    low = int((failed < 0.1).sum())
    ax.set_title(f"{name}: {low} of {len(failed)} failures below 0.1")
axes[0, 0].legend(
    handles=[
        Patch(color=BLUE, alpha=0.55, label="no failure"),
        Patch(color=ORANGE, alpha=0.55, label="failure"),
    ],
    loc="upper center",
)
for ax in axes[1]:
    ax.set_xlabel("Predicted failure probability")
for ax in axes[:, 0]:
    ax.set_ylabel("Density (log scale)")
fig.tight_layout()
plt.show()

### Confusion matrices at the candidate operating points

For one model (change `MODEL` to look at another), the counts at the threshold
with the best F1 and at the three recall targets. Each cell shows the number of
rows; the shading shows the share of the row's true class, so the dark diagonal
means most rows of each class are classified correctly.

In [ ]:
MODEL = "random_forest"

at_targets = compare_at_recall_targets(oof, DEFAULT_TARGETS)
points = [("best F1", best.loc[MODEL])]
points += [
    (f"{target:.0%} recall", at_targets.loc[(target, MODEL)])
    for target in DEFAULT_TARGETS
]
shade = LinearSegmentedColormap.from_list("shade", [SURFACE, BLUE])

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (title, row) in zip(axes, points, strict=True):
    counts = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=int)
    ax.imshow(counts / counts.sum(axis=1, keepdims=True), cmap=shade, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{counts[i, j]:,}",
                ha="center",
                va="center",
                fontsize=13,
                color=INK if counts[i, j] / counts[i].sum() < 0.6 else SURFACE,
            )
    ax.set_xticks([0, 1], ["no failure", "failure"])
    ax.set_yticks([0, 1], ["no failure", "failure"])
    ax.set_xlabel("Predicted")
    ax.set_title(
        f"{title}\nthreshold {row['threshold']:.3f}, precision {row['precision']:.3f}"
    )
    ax.grid(False)
axes[0].set_ylabel("Actual")
fig.suptitle(MODEL, x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 4. Try any threshold

Pick a model and drag the threshold to see the counts and scores on the 8,000
out-of-fold predictions. (The slider works while the notebook is running, and is
not kept when it is saved without outputs.)

In [ ]:
def show_threshold(model, threshold):
    frame = oof[model]
    flagged = (frame["probability"] >= threshold).to_numpy()
    failure = (frame["label"] == 1).to_numpy()
    tp = int((flagged & failure).sum())
    fp = int((flagged & ~failure).sum())
    fn = int((~flagged & failure).sum())
    tn = int((~flagged & ~failure).sum())
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn)
    f1 = 2 * tp / (2 * tp + fp + fn)
    print(
        f"{model}, threshold {threshold:.3f}: flags {tp + fp:,} of {len(frame):,} rows"
    )
    print(f"precision {precision:.3f}   recall {recall:.3f}   F1 {f1:.3f}")
    print(f"correct alerts {tp}, false alarms {fp}, missed failures {fn}")
    print(f"correct non-alerts {tn:,}")


_ = widgets.interact(
    show_threshold,
    model=list(oof),
    threshold=widgets.FloatLogSlider(
        value=0.5,
        base=10,
        min=-2.3,
        max=0,
        step=0.02,
        description="threshold",
        readout_format=".3f",
        continuous_update=False,
    ),
)

## 5. The numbers behind the plots

In [ ]:
at_targets.round(3)

In [ ]:
best.round(3)

## 6. What the figures show

- **Random forest is the best model at every operating point.** It was only tied
  with gradient boosting on PR-AUC, but at a fixed recall it has the fewest false
  alarms, and the gap widens as the recall target rises.
- **There is a knee near 83% recall.** Up to there, the best models find failures
  with almost no false alarms. Beyond it, each extra point of recall costs many
  false alarms, and 90% recall means about one alert in two is false for the
  random forest.
- **ROC curves hide this.** The four ROC curves are almost identical, while the
  precision-recall curves separate the models and show the knee.
- **Some failures are invisible to every model.** About 35 to 40 of the 271
  failures get a probability below 0.1 from every model. These match the failures
  the data generation makes random (tool wear failures inside 200-240 minutes,
  random failures, and failures with no recorded cause), so they cannot be caught
  without flagging many normal machines.
- **Thresholds are not comparable across models.** Class-weighted models such as
  gradient boosting produce inflated probabilities, so their best thresholds are
  much higher (0.86) than the random forest's (0.48). Compare at the same recall.
- **Tuning helps modestly.** Extra trees gains the most from tuning; gradient
  boosting gains almost nothing.
- **Caveat.** The thresholds are chosen on the same out-of-fold predictions they
  are evaluated on, so they are slightly optimistic, and differences of a few
  false alarms between models are within noise.